# Минимизация эмпирического риска: проверяем руками

Ноутбук к теме [`lesson.md`](lesson.md). Здесь проверяется каждая из трёх подмен, на
которых стоит ERM: выборка вместо распределения (несмещённость, концентрация и
проклятие победителя), удобная потеря вместо нужной (лестница 0-1 и суррогаты),
и «куда пришёл алгоритм» вместо минимума (неявное смещение градиентного спуска).
Заодно — гребневая регрессия как MAP с проверкой $\lambda = \sigma^2/\tau^2$.

In [1]:
import warnings

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from sklearn.linear_model import LogisticRegression, Ridge

warnings.filterwarnings("ignore")

C_BG, C_INK, C_GRAY = "#faf9f5", "#141413", "#b0aea5"
C_PANEL, C_ORANGE, C_BLUE, C_GREEN = "#e8e6dc", "#d97757", "#6a9bcc", "#788c5d"
plt.rcParams.update({
    "figure.facecolor": C_BG, "axes.facecolor": C_BG, "axes.edgecolor": C_GRAY,
    "text.color": C_INK, "xtick.color": C_INK, "ytick.color": C_INK,
    "axes.labelcolor": C_INK, "font.size": 11, "legend.frameon": False,
})

## 1. Эмпирический риск — несмещённая оценка истинного

Берём **одну заранее зафиксированную** модель и смотрим, как ведёт себя её
эмпирический риск на выборках разного размера.

In [2]:
rng = np.random.default_rng(0)
N, d = 300_000, 5
X = rng.normal(size=(N, d))
y = (X @ rng.normal(size=d) + 0.8 * rng.normal(size=N) > 0).astype(int)

w_fixed = rng.normal(size=d)                      # модель выбрана ДО просмотра данных
err = (X @ w_fixed > 0).astype(int) != y
R_true = err.mean()
print(f"истинный риск модели (по 300 000 объектам): {R_true:.4f}\n")

print(f"{'n':>7}{'E[R̂] − R':>12}{'std(R̂)':>10}{'теория':>10}{'P(|R̂−R|>0.1)':>16}{'Хёфдинг':>10}")
for n in (10, 30, 100, 300, 1000, 3000, 10000):
    Rhat = np.array([err[rng.integers(0, N, n)].mean() for _ in range(4000)])
    hoeff = min(1.0, 2 * np.exp(-2 * n * 0.1 ** 2))
    print(f"{n:>7}{Rhat.mean() - R_true:>+12.4f}{Rhat.std():>10.4f}"
          f"{np.sqrt(R_true * (1 - R_true) / n):>10.4f}"
          f"{np.mean(np.abs(Rhat - R_true) > 0.1):>16.4f}{hoeff:>10.4f}")

истинный риск модели (по 300 000 объектам): 0.5614

      n   E[R̂] − R   std(R̂)    теория   P(|R̂−R|>0.1)   Хёфдинг
     10     +0.0032    0.1588    0.1569          0.5397    1.0000
     30     +0.0023    0.0901    0.0906          0.2850    1.0000
    100     +0.0005    0.0497    0.0496          0.0430    0.2707
    300     -0.0001    0.0289    0.0286          0.0008    0.0050
   1000     +0.0001    0.0156    0.0157          0.0000    0.0000


   3000     -0.0000    0.0091    0.0091          0.0000    0.0000
  10000     -0.0001    0.0050    0.0050          0.0000    0.0000


Смещение колеблется около нуля, разброс совпадает с $\sqrt{R(1-R)/n}$, а доля выбросов
за 0.1 не превышает границу Хёфдинга $2e^{-2nt^2}$ — как и должно быть. Пока модель
зафиксирована заранее, эмпирический риск честен.

## 2. Проклятие победителя

Теперь выбираем модель **по тем же данным**. Сигнала в данных нет вообще: метки — монетка,
истинный риск любой модели ровно 0.5.

In [3]:
def winners_curse(K, n=50, d=10, reps=200, seed=1):
    rng = np.random.default_rng(seed)
    emp, tru = [], []
    for _ in range(reps):
        X = rng.normal(size=(n, d)); y = rng.integers(0, 2, n)
        W = rng.normal(size=(K, d))
        risks = np.mean((X @ W.T > 0).astype(int) != y[:, None], axis=0)
        k = int(np.argmin(risks))
        emp.append(risks[k])
        Xt, yt = rng.normal(size=(5000, d)), rng.integers(0, 2, 5000)
        tru.append(np.mean((Xt @ W[k] > 0).astype(int) != yt))
    return np.mean(emp), np.mean(tru)

print(f"{'K':>8}{'риск победителя':>18}{'его истинный риск':>20}{'разрыв':>10}{'граница':>10}")
for K in (1, 10, 100, 1000, 10000):
    e, t = winners_curse(K)
    print(f"{K:>8}{e:>18.3f}{t:>20.3f}{t - e:>10.3f}"
          f"{np.sqrt(np.log(max(K, 2)) / (2 * 50)):>10.3f}")

       K   риск победителя   его истинный риск    разрыв   граница


       1             0.498               0.500     0.002     0.083


      10             0.396               0.500     0.104     0.152


     100             0.334               0.499     0.166     0.215


    1000             0.291               0.500     0.209     0.263


   10000             0.259               0.501     0.241     0.303


Ни одна модель ничему не училась — весь эффект даёт **выбор**. Ровно это происходит,
когда перебирают гиперпараметры: лучшая из $K$ конфигураций на валидации оптимистична,
и чем больше перебор, тем сильнее.

## 3. Почему 0-1 потерю не минимизируют напрямую

In [4]:
rng = np.random.default_rng(0)
n = 60
X = rng.normal(size=(n, 2))
y = np.where(X[:, 0] + 0.7 * X[:, 1] + 0.5 * rng.normal(size=n) > 0, 1, -1)

th = np.linspace(0, np.pi, 4000)
W = np.column_stack([np.cos(th), np.sin(th)])
M = (X @ W.T) * y[:, None]                        # отступы: (n, число направлений)
r01 = np.mean(M <= 0, axis=0)
rlog = np.mean(np.log1p(np.exp(-M)), axis=0)

loc = sum(1 for i in range(1, len(r01) - 1)
          if r01[i] < r01[i - 1] and r01[i] <= r01[i + 1])
print(f"0-1 риск вдоль направления: {len(np.unique(r01))} различных значений, "
      f"{loc} локальных минимумов")
print(f"логистический риск: {sum(1 for i in range(1, len(rlog) - 1) if rlog[i] < rlog[i-1] and rlog[i] <= rlog[i+1])} локальных минимумов")

# численный градиент 0-1 риска
eps = 1e-6
i0 = 1200
g01 = (np.mean((X @ np.array([np.cos(th[i0] + eps), np.sin(th[i0] + eps)])) * y <= 0)
       - r01[i0]) / eps
glog = (np.mean(np.log1p(np.exp(-(X @ np.array([np.cos(th[i0] + eps), np.sin(th[i0] + eps)])) * y)))
        - rlog[i0]) / eps
print(f"\nв точке θ = {np.degrees(th[i0]):.1f}°: численная производная 0-1 риска = {g01:.1f}, "
      f"логистического = {glog:.4f}")
print(f"минимум 0-1 при θ = {np.degrees(th[np.argmin(r01)]):.1f}°, "
      f"логистического при θ = {np.degrees(th[np.argmin(rlog)]):.1f}°")

0-1 риск вдоль направления: 47 различных значений, 8 локальных минимумов
логистический риск: 1 локальных минимумов

в точке θ = 54.0°: численная производная 0-1 риска = 0.0, логистического = 0.1602
минимум 0-1 при θ = 16.5°, логистического при θ = 26.3°


Производная 0-1 риска равна нулю почти всюду (между ступеньками) и не существует в точках
разрыва — градиентному методу не за что взяться. Отсюда суррогаты. Проверим, что
минимизация суррогата действительно ведёт 0-1 риск к байесовскому.

In [5]:
def bayes_and_learned(n, seed=0, d=5, noise=0.8):
    rng = np.random.default_rng(seed)
    w_star = rng.normal(size=d)
    def sample(m):
        X = rng.normal(size=(m, d))
        y = (X @ w_star + noise * rng.normal(size=m) > 0).astype(int)
        return X, y
    Xtr, ytr = sample(n)
    Xte, yte = sample(20000)
    bayes = np.mean((Xte @ w_star > 0).astype(int) != yte)   # оптимальное правило известно
    m = LogisticRegression(C=1e6, max_iter=5000).fit(Xtr, ytr)
    return bayes, 1 - m.score(Xte, yte)

print(f"{'n':>7}{'0-1 риск логрегрессии':>24}{'байесовский риск':>20}{'разрыв':>10}")
for n in (20, 50, 200, 1000, 5000, 20000):
    b, r = bayes_and_learned(n)
    print(f"{n:>7}{r:>24.4f}{b:>20.4f}{r - b:>10.4f}")

      n   0-1 риск логрегрессии    байесовский риск    разрыв
     20                  0.3123              0.2397    0.0726
     50                  0.2745              0.2407    0.0338
    200                  0.2376              0.2353    0.0023
   1000                  0.2392              0.2377    0.0015
   5000                  0.2358              0.2361   -0.0003
  20000                  0.2358              0.2357    0.0001


Мы ни разу не оптимизировали долю ошибок, а она сходится к неустранимому минимуму — это и
означает «суррогат классификационно калиброван».

## 4. Регуляризованный ERM — это MAP

Для гауссова правдоподобия с дисперсией $\sigma^2$ и априорного $\mathcal{N}(0, \tau^2 I)$
апостериорное среднее равно $(X^\top X + \frac{\sigma^2}{\tau^2} I)^{-1} X^\top y$ —
то есть гребневой регрессии с $\lambda = \sigma^2/\tau^2$.

In [6]:
rng = np.random.default_rng(7)
n, p = 80, 20
sigma, tau = 0.7, 0.35
beta = rng.normal(scale=tau, size=p)
X = rng.normal(size=(n, p))
y = X @ beta + sigma * rng.normal(size=n)

lam = sigma ** 2 / tau ** 2
w_map = np.linalg.solve(X.T @ X + lam * np.eye(p), X.T @ y)       # апостериорное среднее
w_ridge = Ridge(alpha=lam, fit_intercept=False).fit(X, y).coef_
w_ols = np.linalg.lstsq(X, y, rcond=None)[0]

print(f"λ = σ²/τ² = {lam:.3f}")
print(f"max|MAP − ridge| = {np.abs(w_map - w_ridge).max():.2e}")
print(f"\n{'оценка':<12}{'‖w − β‖':>12}{'‖w‖':>10}")
for name, w in (("МНК (λ=0)", w_ols), ("ridge = MAP", w_ridge), ("истина β", beta)):
    print(f"{name:<12}{np.linalg.norm(w - beta):>12.4f}{np.linalg.norm(w):>10.4f}")

λ = σ²/τ² = 4.000
max|MAP − ridge| = 2.22e-16

оценка           ‖w − β‖       ‖w‖
МНК (λ=0)         0.2971    1.2812
ridge = MAP       0.3052    1.1931
истина β          0.0000    1.2906


## 5. Двойной спуск

Случайные ReLU-признаки, решение минимальной нормы. Интересна точка $p = n$.

In [7]:
rng = np.random.default_rng(3)
n_tr, d0 = 100, 20
beta = rng.normal(size=d0)
Xtr, Xte = rng.normal(size=(n_tr, d0)), rng.normal(size=(4000, d0))
ytr = Xtr @ beta + 0.5 * rng.normal(size=n_tr)
yte = Xte @ beta + 0.5 * rng.normal(size=4000)

print(f"{'p':>7}{'эмпирический риск':>20}{'истинный риск':>18}{'‖коэф‖':>10}")
for p in (10, 50, 90, 100, 110, 200, 1000, 3000):
    W = rng.normal(size=(d0, p)) / np.sqrt(d0)
    Ftr, Fte = np.maximum(Xtr @ W, 0), np.maximum(Xte @ W, 0)
    coef = np.linalg.pinv(Ftr) @ ytr
    mark = "  ← p = n" if p == n_tr else ""
    print(f"{p:>7}{np.mean((Ftr @ coef - ytr) ** 2):>20.2e}"
          f"{np.mean((Fte @ coef - yte) ** 2):>18.2f}{np.linalg.norm(coef):>10.1f}{mark}")

      p   эмпирический риск     истинный риск    ‖коэф‖
     10            1.77e+01             23.93       6.2
     50            3.37e+00             15.25      12.3


     90            2.65e-01             30.55      21.8
    100            7.98e-28             70.40      31.6  ← p = n
    110            3.09e-27             13.40      13.6
    200            1.33e-27              3.10       5.2


   1000            7.01e-28              0.88       1.7
   3000            8.59e-28              0.67       1.0


Норма коэффициентов взрывается ровно в точке интерполяции и падает правее — именно она, а
не число параметров, отвечает за качество.

## 6. Куда приходит градиентный спуск, когда минимума нет

In [8]:
rng = np.random.default_rng(1)
far = rng.normal(loc=[2.2, 2.0], scale=0.5, size=(60, 2))
neg = rng.normal(loc=[-2.0, -2.2], scale=0.5, size=(60, 2))
X = np.vstack([far, [[0.75, 0.15], [0.6, 0.45]], neg, [[-0.5, -0.6], [-0.7, -0.25]]])
y = np.array([1] * 62 + [-1] * 62)

r = minimize(lambda w: 0.5 * w @ w, np.zeros(2), jac=lambda w: w,
             constraints=[{"type": "ineq", "fun": lambda w: y * (X @ w) - 1,
                           "jac": lambda w: y[:, None] * X}],
             method="SLSQP", options={"maxiter": 800, "ftol": 1e-14})
w_mm = r.x / np.linalg.norm(r.x)

w = np.zeros(2)
print(f"{'шаг':>10}{'‖w‖':>9}{'логист. риск':>15}{'0-1 риск':>11}"
      f"{'cos с max-margin':>19}{'отступ':>10}")
for t in range(1, 1_000_001):
    m = y * (X @ w)
    w -= 4.0 * (-(y[:, None] * X * (1 / (1 + np.exp(m)))[:, None]).mean(0))
    if t in (1, 100, 10_000, 1_000_000):
        wn = w / np.linalg.norm(w)
        print(f"{t:>10}{np.linalg.norm(w):>9.2f}"
              f"{np.mean(np.log1p(np.exp(-y * (X @ w)))):>15.6f}"
              f"{np.mean(y * (X @ w) <= 0):>11.3f}{wn @ w_mm:>19.5f}"
              f"{(y * (X @ wn)).min():>10.4f}")
print(f"\nпредельный отступ (максимальный возможный): {(y * (X @ w_mm)).min():.4f}")
print("логистический риск стремится к нулю, но минимума не достигает: ‖w‖ растёт бесконечно")

       шаг      ‖w‖   логист. риск   0-1 риск   cos с max-margin    отступ
         1     5.86       0.000529      0.000            0.96517    0.6421
       100     5.99       0.000476      0.000            0.96719    0.6453


     10000     9.17       0.000043      0.000            0.98969    0.6876


   1000000    15.26       0.000000      0.000            0.99794    0.7120

предельный отступ (максимальный возможный): 0.7285
логистический риск стремится к нулю, но минимума не достигает: ‖w‖ растёт бесконечно


## 7. Что осталось в руках

- Для заранее зафиксированной модели эмпирический риск несмещён, а его разброс падает как
  $1/\sqrt{n}$ — ровно по $\sqrt{R(1-R)/n}$.
- Для модели, **выбранной** по этим же данным, он смещён вниз: на данных без сигнала
  лучшая из 10 000 случайных моделей показывает около 26% ошибок при истинных 50%.
- У 0-1 потери производная равна нулю почти всюду, поэтому минимизируют суррогаты; они
  согласованы с целевой метрикой в пределе, но не на конечной выборке.
- Гребневая регрессия — это в точности MAP с гауссовым априорным распределением и
  $\lambda = \sigma^2/\tau^2$ (совпадение до машинной точности).
- В точке интерполяции $p = n$ норма решения взрывается, а правее — падает вместе с
  истинным риском: двойной спуск.
- Когда минимума не существует, ответ выбирает алгоритм: градиентный спуск уводит решение
  к направлению максимального отступа.